## In this notebook:

#### We concatenate parcellated PET images into region x receptor matrix of densities.
Adapted from Hansen Receptors (https://github.com/netneurolab/hansen_receptors)
Paper: https://www.nature.com/articles/s41593-022-01186-3

### Import Packages

In [5]:
import numpy as np
import pandas as pd
from netneurotools import datasets, plotting
from matplotlib.colors import ListedColormap
from scipy.stats import zscore
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_cammoun2012

In [25]:
import surfer

### Set Variables

In [6]:
path = '/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/'
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'
#figpath = '/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/figures/'

atlas = 'schaefer'

if atlas == 'schaefer':
    scale='scale1000_17'
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17) # This is sklearn.utils._bunch.Bunch. Other altases in this format? *Ask Golia
    nnodes = len(schaefer['labels'])
elif atlas == 'cammoun':
    scale = 'cammoun2012_scale500' #'scale1000_17' #'Harvard_Oxford_sub' 
    folder = 'atl-Cammoun2012_res-500'
    cammoun = fetch_cammoun2012()
    nnodes=1015
elif atlas == 'Harvard_Oxford_sub':
    nnodes = 21


In [4]:
print(path+'data/parcellated/PET_parcellated/'+scale+'/mGluR5_abp_hc73_smart.csv')

/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/cammoun2012_scale500/mGluR5_abp_hc73_smart.csv


### Choose what to analyse

In [23]:
"""
receptors_csv = [path+'data/parcellated/PET_parcellated/'+atlas+scale+'/NMDA_ge179_hc29_galovic.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+scale+'/mGluR5_abp_hc22_rosaneto.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+scale+'/mGluR5_abp_hc28_dubois.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+scale+'/mGluR5_abp_hc73_smart.csv',
                # path+'data/parcellated/PET_parcellated/'+atlas+scale+'/GABAa-bz_flumazenil_hc16_norgaard.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+scale+'/GABAa_flumazenil_hc6_dukart.csv'
                ]
"""

receptors_csv =[path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HT1a_way_hc36_savli.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HT1b_p943_hc22_savli.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HT1b_p943_hc65_gallezot.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HT2a_cimbi_hc29_beliveau.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HT4_sb20_hc59_beliveau.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HT6_gsk_hc30_radhakrishnan.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/5HTT_dasb_hc100_beliveau.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/A4B2_flubatine_hc30_hillmer.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/CB1_omar_hc77_normandin.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/D1_SCH23390_hc13_kaller.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/D2_flb457_hc37_smith.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/D2_flb457_hc55_sandiego.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/DAT_fpcit_hc174_dukart_spect.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/GABAa-bz_flumazenil_hc16_norgaard.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/H3_cban_hc8_gallezot.csv', 
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/M1_lsn_hc24_naganawa.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/mGluR5_abp_hc22_rosaneto.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/mGluR5_abp_hc28_dubois.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/mGluR5_abp_hc73_smart.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/MU_carfentanil_hc204_kantonen.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/NAT_MRB_hc77_ding.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/NMDA_ge179_hc29_galovic.csv',
                # path+'datparcellated/a/PET_parcellated/atlas+'_'+'+scale+'/VAChT_feobv_hc3_spreng.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/VAChT_feobv_hc4_tuominen.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/VAChT_feobv_hc5_bedard_sum.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/VAChT_feobv_hc18_aghourian_sum.csv']


features_csv = [path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/abaPC1_fsavg_values.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/CBF_fsavg_values.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/CBV_fsavg_values.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/CThick_fsavg_values.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/Glucose_fsavg_values.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/O2met_fsavg_values.csv',
                path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/SAaxis_data_scale1000_17N.csv']

       

### Make Receptor Matrices

In [6]:
print(scale) #pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/"+scale+"

cammoun2012_scale500


In [37]:
# combine all the receptors (including repeats)
r = np.zeros([nnodes, len(receptors_csv)])
for i in range(len(receptors_csv)):
    r[:, i] = np.genfromtxt(receptors_csv[i], delimiter=',')



receptor_names = np.array(["5HT1a", "5HT1b", "5HT2a", "5HT4", "5HT6", "5HTT", "A4B2",
                           "CB1", "D1", "D2", "DAT", "GABAa", "H3", "M1", "mGluR5",
                           "MOR", "NET", "NMDA", "VAChT"])
np.save(path+'data/receptor_names_pet.npy', receptor_names)

# make final region x receptor matrix

receptor_data = np.zeros([nnodes, len(receptor_names)])
receptor_data[:, 0] = r[:, 0]
receptor_data[:, 2:9] = r[:, 3:10]
receptor_data[:, 10:14] = r[:, 12:16]
receptor_data[:, 15:18] = r[:, 19:22]

# weighted average of 5HT1B p943
receptor_data[:, 1] = (zscore(r[:, 1])*22 + zscore(r[:, 2])*65) / (22+65)

# weighted average of D2 flb457
receptor_data[:, 9] = (zscore(r[:, 10])*37 + zscore(r[:, 11])*55) / (37+55)

# weighted average of mGluR5 ABP688
receptor_data[:, 14] = (zscore(r[:, 16])*22 + zscore(r[:, 17])*28 + zscore(r[:, 18])*73) / (22+28+73)

# weighted average of VAChT FEOBV
receptor_data[:, 18] = (zscore(r[:, 22])*4 + zscore(r[:, 23]) + zscore(r[:, 24])) / \
                       (4+5+18)

# Make df and add parcel labels
receptor_df = pd.DataFrame(receptor_data, columns=receptor_names)
if atlas == 'schaefer':
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
    labels = schaefer.labels
    labels = [label.decode('utf-8') for label in labels]

receptor_df.index = labels
receptor_df.index.name = 'Parcel'


# Save df
receptor_df.to_csv(path+'results/receptor_data_all_'+scale+'.csv', index=True, header=True)

In [36]:
# OLD JUST GLUTAMATE
r = np.zeros([nnodes, len(receptors_csv)])
print(r.shape)
for i in range(len(receptors_csv)):
    r[:, i] = np.genfromtxt(receptors_csv[i], delimiter=',') 
    # Now we have the receptor data by parcel (row) for all 6 receptor maps (col)

#print(r[:,3])
print(r.shape)

receptor_names = np.array(["NMDA","mGluR5","GABAa"]) #,"D2"
np.save(path+'data/receptor_names_pet.npy', receptor_names)

# make final region x receptor matrix
receptor_data = np.zeros([nnodes, len(receptor_names)])

# NMDA Data
receptor_data[:, 0] = zscore(r[:, 0])
# weighted average of mGluR5 ABP688
receptor_data[:, 1] = (zscore(r[:, 1])*22 + zscore(r[:, 2])*28 + zscore(r[:, 3])*73) / (22+28+73)
# weighted average of GABA 
receptor_data[:, 2] = zscore(r[:, 4])


# Make df and add parcel labels
receptor_df = pd.DataFrame(receptor_data, columns=receptor_names)
if atlas == 'schaefer':
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
    labels = schaefer.labels
    labels = [label.decode('utf-8') for label in labels]
elif atlas == 'cammoun':
    cam = fetch_cammoun2012()
    info = pd.read_csv(cammoun['info'], sep=',')
    info = info[info['scale']== 'scale500']
    hemisphere = info['hemisphere'].values
    labels = info['label'].values + hemisphere
    print(labels)
    
receptor_df.index = labels
receptor_df.index.name = 'Parcel'


# Save df
receptor_df.to_csv(path+'results/receptor_data_'+atlas+scale+'.csv', index=True, header=True)

#print(receptor_df)

(1000, 25)
(1000, 25)


In [23]:
saaxis = features_csv[6]
axis = np.genfromtxt(saaxis, delimiter=',',usecols=2) 
print(axis)

[ nan 541. 324. ... 550. 727. 631.]


In [42]:
# Make matrix with surface data
# combine all the receptors (including repeats)
f = np.zeros([nnodes, len(features_csv)])
feature_names = np.array(['abaPC1','CBF','CBV','CThick','Glucose','O2met','SAaxis']) #,"D2"
np.save(path+'data/feature_names_pet.npy', receptor_names)

# Make feature np.array
for i in range(len(features_csv)):
    if i == 6: # For SAaxis
        sa_array=np.genfromtxt(features_csv[i], delimiter=',',usecols=2)
        sa_array=sa_array[1:]
        print(sa_array.shape)
        f[:,i] = sa_array #np.genfromtxt(sa_array, delimiter=',')
    else:
        f[:, i] = np.genfromtxt(features_csv[i], delimiter=',') 
    
                  
# make final region x feature matrix
feature_data = np.zeros([nnodes, len(feature_names)])
for i in range(len(features_csv)):
    feature_data[:, i] = zscore(r[:, i])

# Make df and add parcel labels
feature_df = pd.DataFrame(feature_data, columns=feature_names)
feature_df.index = labels
feature_df.index.name = 'Parcel'


# Save df
feature_df.to_csv(path+'results/feature_data_all_'+atlas+scale+'.csv', index=True, header=True)


(1000,)


In [13]:
"""
plot receptor data
"""

# colourmaps
cmap = np.genfromtxt(datapath+'data/colourmap.csv', delimiter=',')
cmap_div = ListedColormap(cmap)

# plot each receptor map
if scale == 'scale1000_17':
    annot = datasets.fetch_schaefer2018('fsaverage')['1000Parcels17Networks']
    type(annot)
    print(annot)
    
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot=annot.lh,
                                        rhannot=annot.rh,
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(path+'figures/scale1000_17/surface_receptor_'+receptor_names[k]+'.png')


if scale == 'scale100':
    annot = datasets.fetch_schaefer2018('fsaverage')['100Parcels7Networks']
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot=annot.lh,
                                        rhannot=annot.rh,
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(path+'figures/' + scale + '/surface_receptor_'+receptor_names[k]+'.png')

## NEWWWWWW
if scale == 'cammoun2012_scale500':
    cammoun = datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    print(cammoun)
    #annot = datasets.fetch_schaefer2018('fsaverage')['100Parcels7Networks']
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot = cammoun[0],
                                        rhannot = cammoun[1],
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(figpath + folder + '/surface_receptor_'+receptor_names[k]+'.png')




#receptor_data[:, 2:9] = r[:, 3:10]
print(receptor_data)
print(len(receptor_data))

print(r)

Surface(lh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-L_deterministic.annot', rh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-R_deterministic.annot')


qt.qpa.window: <QNSWindow: 0x12fa076f0; contentView=<QNSView: 0x177c4dcb0; QCocoaWindow(0x600001cec210, window=QWidgetWindow(0x600000edb8a0, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x373a88220; contentView=<QNSView: 0x373a87e20; QCocoaWindow(0x600001cf35a0, window=QWidgetWindow(0x600000ebe700, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x177c68f70; contentView=<QNSView: 0x177c68b70; QCocoaWindow(0x600001cecb00, window=QWidgetWindow(0x600000ea27c0, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These wil

[[-0.02086797  0.16462124 -0.15022788]
 [ 0.36601258 -0.0853896   0.17048341]
 [ 1.17184921  1.01544628  1.31715741]
 ...
 [ 0.85539512 -1.00922289 -1.03965146]
 [ 0.10952881 -0.78241158 -0.97148672]
 [-0.79893758 -4.70604028 -5.10899002]]
1015
[[ 6.30872320e+00  3.60468964e+00  2.77503733e+00  7.55093270e-01
   6.18744121e+01]
 [ 6.75982295e+00  3.13090897e+00  2.62848026e+00  7.97383492e-01
   6.48180109e+01]
 [ 7.69942227e+00  3.83834514e+00  2.92945047e+00  9.25286237e-01
   7.53425800e+01]
 ...
 [ 7.33043923e+00  2.85463526e+00  2.34415652e+00  6.47032100e-01
   5.37109760e+01]
 [ 6.46076483e+00  3.14520325e+00  2.41282438e+00  6.48180838e-01
   5.43366155e+01]
 [ 5.40150000e+00  1.94463151e+00  1.43390741e+00 -4.63217504e-02
   1.63611847e+01]]
